### Bronze layer processing

In [0]:
from pyspark.sql import functions as F
from delta.tables import DeltaTable 

In [0]:
%run /Users/gainrakesh2001@gmail.com/consolidated_pipeline/1_setup/utilities

In [0]:
print(bronze_schema,silver_schema,gold_schema)

In [0]:
dbutils.widgets.text("catalog","fmcg","Catalog")
dbutils.widgets.text("data_source","products","Data Source")

In [0]:
 # created widgets becuase dynamically change the catalog,data_source
# suppose this can run different environment like dev ,test,prod e.g (dev_fmcg,test_fmcg,prod_fmcg) and (prod_fmcg,test_fmcg,dev_fmcg) etc
catalog = dbutils.widgets.get("catalog")
data_source = dbutils.widgets.get("data_source")

base_path = f"/Volumes/workspace/default/sportsbar_raw/{data_source}/"
print(catalog,data_source)
print(base_path)

In [0]:
# loading the data into in df
df = spark.read.format("csv")\
        .option("header",True)\
        .option("inferSchema",True)\
        .load(base_path)

df = df.withColumn("read_timestamp",F.current_timestamp())
df = df.select("*","_metadata.file_name","_metadata.file_size")
display(df)

In [0]:
df.printSchema()

In [0]:
df.write\
    .format("delta")\
    .option("delta.enableChangeDataFeed","true")\
    .mode("overwrite")\
    .saveAsTable(f"{catalog}.{bronze_schema}.{data_source}")

### Silver Processing

In [0]:
df_bronze = spark.sql(f"""select * from {catalog}.{bronze_schema}.{data_source}""")
df_bronze.show()

### Drop duplicate 

In [0]:
#Check the duplicate
df_duplicate = df_bronze.groupBy("product_id").count().withColumnRenamed("count", "duplicate_customer").filter(F.col("duplicate_customer") > 1)
display(df_duplicate)

In [0]:
# remove the duplicate and save the data in silver table
print("Row before duplicate dropped: ", df_bronze.count())
df_silver = df_bronze.dropDuplicates(["product_id"])
print("Row after duplicate dropped: ", df_silver.count())

Title case fix

(energy bars - > Energy Bar,protien bars ---- > Protien Bars etc)

In [0]:
df_silver.select("category").distinct().show()

In [0]:
#title case fix
df_silver = df_silver.withColumn(
    "category",
    F.when(F.col("category").isNull(),None)
        .otherwise(F.initcap("category"))
)

In [0]:
df_silver.select("category").distinct().show()

3. Fix Speling Mistake for Protien

In [0]:
# Replace 'protien -> protein in both product_name and catagory

df_silver = df_silver.withColumn(
    "product_name", F.regexp_replace(F.col("product_name"), "(?i)Protien", "Protein")
).withColumn(
    "category", F.regexp_replace(F.col("category"), "(?i)Protien", "Protein")
)


In [0]:
df_silver.select("category").distinct().show()

### standardizing Customer Attributes to Match Parent Company Data Model

In [0]:
# Add dividion column
df_silver = (
    df_silver.withColumn(
        "division",
        F.when(F.col("category") == "Energy Bars" , "Nutrition Bars")
        .when(F.col("category") == "Protein Bars" , "Protein bars")
        .when(F.col("category") == "Granola & Cereals" , "Breakfast Foods")
        .when(F.col("category") == "Recovery Dairy" , "Dairy & Recovery")
        .when(F.col("category") == "Healthy Snacks" , "Healthy Snacks")
        .when(F.col("category") == "Electrolyte Mix" , "Hydration & Electrolytes")
        .otherwise("Other")
    )
)


# Variant column
df_silver = df_silver.withColumn(
    "variant" , 
    F.regexp_extract(F.col("product_name") , "\((.*?)\)", 1)
)

In [0]:
# Create new column : product_code

# Invalid product_id are replaced with a fallback value to avoid losing fact record and ensure downstream join reamin consistent
df_silver = df_silver.withColumn("product_code", F.sha2(F.col("product_name").cast("string") , 256)) # It will create a new coulmn that has new product_code that is unique and it's surrogate key and it will be the primary key

df_silver = df_silver.withColumn(
        "product_id" , F.when(F.col("product_id").cast("string").rlike("^[0-9]+$"), F.col("product_id").cast("string"))
                            .otherwise(F.lit(999999).cast("string")))

df_silver = df_silver.withColumnRenamed("product_name","product")

In [0]:
display(df_silver)

In [0]:
df_silver = df_silver.select("product_code","division","category","product","variant","product_id","read_timestamp","file_name","file_size")

In [0]:
display(df_silver)

In [0]:
df_silver.write\
        .format("delta")\
        .option("delta.enableChangeDataFeed", "true")\
        .option("mergeSchema","true")\
        .mode("overwrite")\
        .saveAsTable(f"{catalog}.{silver_schema}.{data_source}")

### Gold layer Processing 

In [0]:
df_silver = spark.sql(f"""
                      select * from {catalog}.{silver_schema}.{data_source} 
                      """)
df_gold = df_silver.select("product_code","product_id","division","category","product","variant")
df_gold.show()

In [0]:
df_gold.write\
                .format("delta")\
                .option("delta.enableChangeDataFeed", "true")\
                .option("mergeSchema","true")\
                .mode("overwrite")\
                .saveAsTable(f"{catalog}.{gold_schema}.sb_dm_{data_source}") # this gold data for sports bar company that's why sb_dm


### Merging Data source with parent

In [0]:
# For this code merging the parent company data with the child company data in dim_customer table (gold table)
delta_table = DeltaTable.forName(spark, "fmcg.gold.dim_products")
df_child_products = spark.sql(""" select product_code,division,category,product , variant from fmcg.gold.sb_dm_products""")


In [0]:
# Upsert the data into the delta table
delta_table.alias("target").merge(
    source= df_child_products.alias("source"),
    condition = "target.product_code = source.product_code"
).whenMatchedUpdate(
    set = {
        "division": "source.division",
        "category": "source.category",
        "product": "source.product",
        "variant": "source.variant"
    }
).whenNotMatchedInsert(
    values={
        "product_code": "source.product_code",
        "division": "source.division",
        "category": "source.category",
        "product": "source.product",
        "variant": "source.variant"
    }
).execute()